# Import

In [29]:
!rm -rf logs/ # clear logs
!rm -rf optimizer_output/

In [30]:
import numpy as np
import logging

from pathlib import Path
from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup, Nevergrad_Spice_Base_Optimizer
from symxplorer.designer_tools.domains      import OptimizationLogEntry
from symxplorer.logging                     import setup_loggers

setup_loggers()
logger = logging.getLogger("SymXplorer.jupyter")
# Force optimizer logging to INFO
logging.getLogger("SymXplorer.optimizer").setLevel(logging.INFO)

logger.info("Starting the experiment!")


06:51:02 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
06:51:02 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-02_06-51-02.log
06:51:02 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
06:51:02 - SymXplorer.jupyter: [INFO] Starting the experiment!


# Visualizer Function

In [31]:
def visualize_loss_fn(optimizer_cls: Nevergrad_Spice_Base_Optimizer.__class__, setup_obj: Project_Setup, spec_name: str, array_of_values) -> None:
    logger = logging.getLogger("SymXplorer.visualize_loss_fn")
    logger.info(f"using {len(array_of_values)} points")
    logger.info(f"\tTarget: {setup_obj.optimizer_config.target_specs.get_target_by_name(spec_name)}")

    dummy_circuit_optimizer_obj = optimizer_cls(spicelib_wrapper=None, setup_obj=setup_obj)
    dummy_circuit_optimizer_obj.verbose = False
    for val in array_of_values:
        loss, fit_summary = dummy_circuit_optimizer_obj.compute_fitness(performance_array={spec_name : val})
        dummy_circuit_optimizer_obj.optimization_log.append({
                "metric_value": None,
                "fit_summary": fit_summary,
                "params": None,
                "log": None
            })

    dummy_circuit_optimizer_obj.plot_loss_value_by_spec(spec_name=spec_name, show = True)

# Playground

## Load the project setup

In [32]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

06:51:02 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
06:51:02 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=150, random_seed=48
06:51:02 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
06:51:02 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
06:51:02 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
06:51:02 - SymXplorer.domains: [INFO] 	Number of target specs: 4
06:51:02 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=10.0, enable=True, description=Center frequency)
06:51:02 - SymXplorer.domains: [INFO] 		- TargetSpec(name=q, target=10, range=1.00e+01 tolerance=2, goal=exact, sim_type=ac, enable

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06), 'max_ind_size': np.float64(1e-08), 'min

## Visualize the loss function for each spec

In [33]:
# Should match the Project Setup
spec_name = 'gain_db'

# Array of values
min_val  = np.float64(-200.0)
max_val  = np.float64(135)
points_per_unit = 10

vals = np.linspace(min_val, max_val, int((max_val-min_val) * (points_per_unit)))  

# Visualize the loss function
visualize_loss_fn(optimizer_cls=Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, setup_obj=PROJECT_SETUP, spec_name=spec_name, array_of_values=vals)

06:51:02 - SymXplorer.visualize_loss_fn: [INFO] using 3350 points
06:51:02 - SymXplorer.visualize_loss_fn: [INFO] 	Target: TargetSpec(name=gain_db, target=40, range=1.00e+02 tolerance=5, goal=exceed, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=10.0, enable=True, description=gain in dB at fc)
06:51:02 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 4 target specs
06:51:02 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 8.258684551194573
06:51:02 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


In [34]:
# Should match the Project Setup
spec_name = 'fc'

# Array of values
min_val  = np.float64(1e4)
max_val  = np.float64(1e8)

total_points = 1e2

vals = np.linspace(min_val, max_val, int(total_points))  

# Visualize the loss function
visualize_loss_fn(optimizer_cls=Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, setup_obj=PROJECT_SETUP, spec_name=spec_name, array_of_values=vals)

06:51:02 - SymXplorer.visualize_loss_fn: [INFO] using 100 points
06:51:02 - SymXplorer.visualize_loss_fn: [INFO] 	Target: TargetSpec(name=fc, target=100e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=10.0, enable=True, description=Center frequency)
06:51:02 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 4 target specs
06:51:02 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 4.218579043215518
06:51:02 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


In [35]:
# Should match the Project Setup
spec_name = 'pm'

# Array of values
min_val  = np.float64(0)
max_val  = np.float64(180)

total_points = 180

vals = np.linspace(min_val, max_val, int(total_points))  

# Visualize the loss function
visualize_loss_fn(optimizer_cls=Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, setup_obj=PROJECT_SETUP, spec_name=spec_name, array_of_values=vals)

06:51:02 - SymXplorer.visualize_loss_fn: [INFO] using 180 points
06:51:02 - SymXplorer.visualize_loss_fn: [INFO] 	Target: TargetSpec(name=pm, target=70, range=4.50e+01 tolerance=10, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=10.0, enable=True, description=phase margin)
06:51:02 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 4 target specs
06:51:02 - SymXplorer.plotter: [INFO] 	min loss 0.0; max loss 8.044548002984016
06:51:02 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...
